# Ingestion PDF

## Loader sait : 

- où trouver le fichier 
- comment l'ouvrir 
- comment lire sas structure interne
- et comment en extraire le texte

## PyPDFLoader

In [62]:
from langchain_community.document_loaders import PyPDFLoader 

In [63]:
pdf_path = "../data/pdf_files/rapport-activite_credit_mutuelle.pdf"

In [64]:
loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load()

In [65]:
print("Nombre de pages :", len(pdf_docs))
print(pdf_docs[3].page_content[:500])
print(pdf_docs[0].metadata)

Nombre de pages : 79
Le modèle mutualiste reste-t-il un atout et un levier pour 
le développement présent et futur du Groupe ?
Daniel Baal : Oui.et.plus.que.jamais.!.Il.repose.sur.des.
valeurs.de.proximité,.de.solidarité.et.de.gouvernance.
démocratique,. qui. font.notre.singularité. et.notre.
force. face. aux. défis. économiques. et. sociétaux.
actuels.. Dans. un. environnement. en. constante .
évolution,.marqué.par.la.digitalisation,.la.transition.
écologique.et.les.attentes.croissantes.des.clients.en.
matière.de.r
{'producer': 'iLovePDF', 'creator': 'Adobe InDesign 20.4 (Macintosh)', 'creationdate': '2025-07-18T16:01:37+02:00', 'moddate': '2025-07-18T16:31:45+02:00', 'trapped': '/False', 'source': '../data/pdf_files/rapport-activite_credit_mutuelle.pdf', 'total_pages': 79, 'page': 0, 'page_label': '1'}


## Ingestion avancée avec PyMUPDFLoader

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = "../data/pdf_files/rapport-activite_credit_mutuelle.pdf"
loader2 = PyMuPDFLoader(pdf_path)
pdfmu_docs = loader2.load()

print("Nombre de pages :", len(pdfmu_docs))
print(pdfmu_docs[3].page_content[:500])
print(pdfmu_docs[3].metadata)

/tmp/ipykernel_93442/3440294139.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


ValueError: File path ../data/pdf_files/rapport-activite_credit_mutuelle.pdf is not a valid file or url

## Fonction de nettoyage

In [67]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # 0. Normaliser les caractères unicode (très important pour les points PDF)
    text = unicodedata.normalize("NFKC", text)

    # 1. Supprimer tous les points et assimilés (Unicode category: Po = punctuation other)
    text = re.sub(r"[.\u2022\u2027\u2219\u00B7\u2043\u25CF\u25E6\u2024\uFE52\uFF0E]+", " ", text)

    # 2. Supprimer les traits d’union PDF
    text = re.sub(r"-\s+", "", text)

    # 3. Supprimer les multiples espaces
    text = re.sub(r"\s+", " ", text)

    # 4. Supprimer les séparateurs horizontaux
    text = re.sub(r"[_=]{3,}", " ", text)

    # 5. Supprimer les numéros de page
    text = re.sub(r"Page\s*\d+\s*/\s*\d+", " ", text, flags=re.IGNORECASE)

    # 6. Nettoyage final
    text = text.strip()
    return text


## Appliquer le nettoyage 

In [68]:
cleaned_docs = []

for doc in pdf_docs:
    cleaned = clean_pdf_text(doc.page_content)
    doc.page_content = cleaned
    cleaned_docs.append(doc)

## Avant/Après

In [69]:
print("AVANT :")
print(pdfmu_docs[11].page_content[:300])

print("\nAPRÈS :")
print(cleaned_docs[11].page_content[:300])

AVANT :
Une banque qui  
appartient à ses clients,  
ça change tout !
Les forces du modèle Crédit Mutuel
Le sociétaire : maillon central  
de la chaîne des décisions et des actions
Le réseau Crédit Mutuel en 2024
★ 2 793 points de vente
★ 1 911 caisses locales
★ 12,6 millions de clients dont  9,2 million

APRÈS :
Une banque qui appartient à ses clients, ça change tout ! Les forces du modèle Crédit Mutuel Le sociétaire : maillon central de la chaîne des décisions et des actions Le réseau Crédit Mutuel en 2024 ★ 2 793 points de vente ★ 1 911 caisses locales ★ 12,6 millions de clients dont 9,2 millions de socié


## CHUNKING

LES VALEURS OPTIMALES :

- chunk_size = entre 1000 et 1200 caractères

- overlap = entre 100 et 200 caractères

Par exemple :

chunk 1 : phrase A + début de phrase B

chunk 2 : fin de phrase B + phrase C

## IMPLEMENTATION DU CHUNCKING

In [70]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [71]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200,
    chunk_overlap = 150,
    separators = ["\n\n", "\n", ".", " ", ""]
)

In [72]:
chunks = splitter.split_documents(cleaned_docs)

print("Nombre de chunks :", len(chunks))
print(chunks[3].page_content)

Nombre de chunks : 207
DANIEL BAAL Président de la Confédération Nationale du Crédit Mutuel ISABELLE FERRAND Directrice générale de la Confédération Nationale du Crédit Mutuel Portés par le cercle vertueux du modèle mutualiste, nous concilions performance et engagement Le mot de la présidence et de la direction générale Le mot de la présidence et de la direction générale Sommaire 3 RAPPORT D’ACTIVITÉS / Édition 2025


In [78]:
for i, c in enumerate(chunks[:5]): 
    print(f"---- CHUNK{i} -----")
    print(c.page_content[:400])
    print()

---- CHUNK0 -----
2020 2525 ÉDITION GROUPE CRÉDIT MUTUEL 2024 Rapport d’activités 2025

---- CHUNK1 -----
Sommaire LE MOT DE LA PRÉSIDENCE ET DE LA DIRECTION GÉNÉRALE p 3 LES TEMPS FORTS 2024 p 5 0505 Jamais à court d’idées p 38 0101 Toujours plus coopératifs p 7 0202 Des performances solides p 18 0303 À la hauteur de nos engagements p 28 0606 Sur tous les territoires p 42 0707 La bancassurance de proximité p 48 0808 Innovation p 69 0404 DOSSIER SPÉCIAL : LES ÉNERGIES COOPÉRATIVES À L’HONNEUR p 34 2 R

---- CHUNK2 -----
Dans un cadre d’incertitudes économique et politique, comment analysez-vous les résultats de 2024 ? Daniel Baal : Dans ce contexte complexe et instable, le groupe Crédit Mutuel réalise d’excellents résultats, avec un bénéfice net frôlant son record de 2023 à 4,5 milliards d’euros Nous confirmons ainsi notre place de troisième groupe bancaire français Ces performances soulignent l’importance de la 

---- CHUNK3 -----
DANIEL BAAL Président de la Confédération Nationale du

In [94]:
print(chunks[15].page_content)

Une banque différente par son engagement pour tous & pour chacun À l’image du développement accéléré de l’intelligence artificielle, notre monde se transforme profondément et rapidement, modifiant les équilibres globaux comme les repères individuels Or s’il ouvre de nouvelles sources d’inspiration, le progrès n’a de sens que par son utilité dans le monde réel, au service de tous Banque indépendante et mutualiste, le Crédit Mutuel évolue au cœur de ces changements avec un objectif prioritaire : continuer, comme il l’a toujours fait, à agir dans l’intérêt collectif et à proposer des solutions ajustées aux besoins de chacun Plus protecteur Depuis toujours, protéger les personnes et les biens est au cœur de notre activité Ce n’est donc pas vraiment un hasard si c’est le Crédit Mutuel qui a inventé le concept de bancassurance dès 1970 pour toujours mieux accompagner ses différentes clientèles et couvrir les risques auxquels elles sont exposées Plus solidaire La santé ne doit pas être un fre

In [95]:
print(chunks[16].page_content)

laquelle le Crédit Mutuel n’a pas hésité à supprimer les formalités médicales pour ses clients fidèles qui empruntent afin d’acheter leur résidence principale Plus engagé Prouver que nos engagements sont bel et bien concrets fait partie du pacte de confiance que nous avons noué avec tous nos clients Afin de mesurer les impacts environnementaux et sociétaux dans sa performance globale, le Crédit Mutuel Arkéa quantifie en euros ses impacts extra-financiers Plus responsable Depuis 2023, Crédit Mutuel Alliance Fédérale affecte chaque année 15 % de son résultat net au financement de projets de transformation sociétale et environnementale Ce dividende sociétal prouve une fois encore combien notre développement et nos ambitions restent indissociables de notre sens des responsabilités En 2024, 574 millions d’euros ont ainsi été mobilisés pour soutenir des entrepreneurs engagés pour le climat, porter des offres de bancassurance inclusives et accompagner les associations locales comme les grande

# Retenez simplement ceci : 
- Le chunking découpe le texte 
- l'overlap recolle les idées 

## CONCLUSION

In [ ]:
Vous venez d’apprendre le pipeline complet de préparation d’un PDF :

✔ comment charger un PDF simple
✔ comment gérer un PDF complexe
✔ pourquoi un PDF n’est pas du texte brut
✔ comment nettoyer proprement le texte
✔ comment recoller les mots
✔ comment supprimer les artefacts
✔ comment chunker proprement
✔ et pourquoi l’overlap est essentiel